# mT0-large QLoRA Fine-tuning â€” RAG for Multilingual Health QA

End-to-end pipeline for fine-tuning mT0-large on your RTX 4050 (6 GB VRAM), overnight.

**Why mT0-large over mT5-large?**
- Instruction-tuned (no sentinel-token emission issues mT5 has)
- Same architecture and size (~1.2B params)
- fp16-stable (unlike mT5)
- Trained on multilingual instruction data including African languages

**Why this fits in 6 GB:**
- **4-bit QLoRA** quantization: base model ~700 MB instead of ~5 GB (fp32) or ~2.5 GB (fp16)
- **LoRA adapters only** trainable (~5 MB params)
- Batch size 1 + gradient accumulation 16 (effective batch 16)
- Gradient checkpointing
- Short sequences (input 384, output 256)

**Time budget:** ~10-14 hours for 2 epochs on ~30K examples. Overnight run.

## What you need before running

1. **mT0-large on disk** â€” set `MT0_PATH` below to its folder
2. **Fine-tuned BGE-M3 LoRA on disk** â€” set `BGE_M3_LORA_DIR` below
3. **Data files** â€” `Train.csv`, `Val.csv`, `Test.csv` in `DATA_DIR`

## 1 â€” Install dependencies (Windows-friendly versions)

In [1]:
# Run once. bitsandbytes 0.43+ has good Windows support.
# !pip install -U "transformers>=4.46.0,<5.0.0" "peft>=0.12.0" "datasets>=3.0.0" \
#     "bitsandbytes>=0.43.0" "accelerate>=1.0.0" "sentence-transformers>=3.0" \
#     "scikit-learn" "rouge-score" "sentencepiece"

# If bitsandbytes install gives trouble on Windows, use:
# !pip install bitsandbytes-windows  (older but reliable)
# OR the official Windows wheel from https://github.com/jllllll/bitsandbytes-windows-webui

print('Skipping install â€” uncomment above on first run.')

Skipping install â€” uncomment above on first run.


## 2 â€” Imports & Reproducibility

In [2]:
import os
# bitsandbytes 0.49.2 ships CUDA 13.0 binaries, while this kernel uses torch+CUDA 13.2.
# The 13.0 bnb binary works on this GPU and keeps QLoRA/NF4 loading in 4-bit.
os.environ.setdefault('BNB_CUDA_VERSION', '130')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Let Ada/Ampere tensor cores use TF32 where applicable; QLoRA weights stay 4-bit/NF4.
torch_matmul_precision = 'high'

import gc, re, time, random, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
torch.set_float32_matmul_precision(torch_matmul_precision)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
from tqdm.auto import tqdm

from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
from peft import (
    LoraConfig, TaskType, get_peft_model,
    PeftModel, prepare_model_for_kbit_training,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    print(f'bitsandbytes CUDA override: {os.environ.get("BNB_CUDA_VERSION")}')

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU  (6.4 GB)
bitsandbytes CUDA override: 130


## 3 â€” Paths & Config

In [3]:
# === EDIT THESE PATHS ===
MT0_PATH        = r'C:\Users\Papa Offei\Documents\lalang\mt0'                    # your local mT0-large folder
BGE_M3_LORA_DIR = r'C:\Users\Papa Offei\Documents\lalang\Bgem3-finetune\bge-m3-health-qa\final'      # your local BGE-M3 LoRA folder
DATA_DIR        = Path(r'C:\Users\Papa Offei\Documents\lalang')      # contains Train.csv, Val.csv, Test.csv

# === Outputs ===
OUT_DIR    = Path('./mt0-rag-finetuned')
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION = Path('./submission_mt0_rag.csv')

# === Columns (match challenge convention) ===
QCOL, ACOL, GCOL, IDCOL = 'input', 'output', 'subset', 'ID'

CONFIG = {
    # === Experiment mode ===
    # Quick mode is for deciding whether mT0+RAG is promising before spending a full-day run.
    # Set quick_experiment=False for the full training/eval pass.
    'quick_experiment'      : True,
    'quick_train_rows'      : 8000,      # stratified across subsets; ~3-4h on the RTX 4050 from current speed
    'quick_eval_rows'       : 800,       # stratified validation sample for generation/ROUGE check
    'eval_during_training'  : False,     # saves time; do generation-based eval after training instead

    # === Retrieval ===
    'top_k_context'         : 3,         # keep 3; reducing this is a quality risk
    'cand_question_max_chars': 180,       # retrieved questions are context labels; keep them short
    'cand_answer_max_chars' : 320,       # small trim from 350, still enough answer context

    # === Model ===
    'mt0_path'              : MT0_PATH,
    'max_input_length'      : 384,       # keep this; prompt/context truncation risk is higher below 384
    'max_target_length'     : 240,       # conservative speed trim from 256; avoids cutting too many long answers

    # === Training (RTX 4050 6 GB safe defaults) ===
    'num_epochs'            : 1,         # quick proof run; use 2 for a final run if val improves
    'batch_size'            : 1,         # do NOT increase on 6 GB VRAM
    'gradient_accumulation' : 16,        # preserves the earlier effective batch size
    'learning_rate'         : 3e-4,      # higher for LoRA
    'warmup_ratio'          : 0.05,
    'weight_decay'          : 0.01,
    'gradient_checkpointing': True,

    # === LoRA ===
    'lora_r'                : 16,
    'lora_alpha'            : 32,
    'lora_dropout'          : 0.05,
    'lora_target_modules'   : ['q', 'v'],   # attention only - keep memory low

    # === Inference ===
    'gen_batch_size'        : 4,
    'num_beams'             : 1,         # greedy for quick evaluation; use 4 for final submission generation
    'final_num_beams'       : 4,
    'no_repeat_ngram_size'  : 3,
    'length_penalty'        : 1.0,
}
print(json.dumps({k: v for k, v in CONFIG.items() if not isinstance(v, list)}, indent=2))


{
  "quick_experiment": true,
  "quick_train_rows": 8000,
  "quick_eval_rows": 800,
  "eval_during_training": false,
  "top_k_context": 3,
  "cand_question_max_chars": 180,
  "cand_answer_max_chars": 320,
  "mt0_path": "C:\\Users\\Papa Offei\\Documents\\lalang\\mt0",
  "max_input_length": 384,
  "max_target_length": 240,
  "num_epochs": 1,
  "batch_size": 1,
  "gradient_accumulation": 16,
  "learning_rate": 0.0003,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "gradient_checkpointing": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "gen_batch_size": 4,
  "num_beams": 1,
  "final_num_beams": 4,
  "no_repeat_ngram_size": 3,
  "length_penalty": 1.0
}


## 4 â€” Load Data

In [4]:
train = pd.read_csv(DATA_DIR / 'Train.csv')
val   = pd.read_csv(DATA_DIR / 'Val.csv')
test  = pd.read_csv(DATA_DIR / 'Test.csv')

for df in (train, val, test):
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna('').astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna('').astype(str).str.strip()
train = train[(train[QCOL] != '') & (train[ACOL] != '')].reset_index(drop=True)
val   = val[(val[QCOL] != '') & (val[ACOL] != '')].reset_index(drop=True)

print(f'train: {len(train)}  val: {len(val)}  test: {len(test)}')

train: 29814  val: 6686  test: 2618


## 5 â€” ROUGE Scorer (whitespace tokenizer)

In [5]:
class WhitespaceTokenizer:
    def tokenize(self, t):
        return [] if t is None else str(t).strip().split()

_SCORER = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False,
)

def rouge_metrics(preds, refs):
    if not preds:
        return {'rouge1_f1': 0.0, 'rougeL_f1': 0.0}
    r1, rl = [], []
    for p, r in zip(preds, refs):
        s = _SCORER.score(str(r), str(p))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}

def per_subset_report(preds, refs, subs, label):
    sub = np.array(subs); rows = []
    for s in sorted(np.unique(sub)):
        m = sub == s
        prs = [preds[i] for i in range(len(preds)) if m[i]]
        rfs = [refs[i]  for i in range(len(refs))  if m[i]]
        sc = rouge_metrics(prs, rfs)
        rows.append({'subset': s, 'n': int(m.sum()),
                     f'{label}_r1': round(sc['rouge1_f1'], 4),
                     f'{label}_rL': round(sc['rougeL_f1'], 4)})
    overall = rouge_metrics(preds, refs)
    rows.append({'subset': 'OVERALL', 'n': len(preds),
                 f'{label}_r1': round(overall['rouge1_f1'], 4),
                 f'{label}_rL': round(overall['rougeL_f1'], 4)})
    return pd.DataFrame(rows)

## 6 â€” Retrieve Top-K Candidates Using Your Fine-tuned BGE-M3

We do this **first**, before loading mT0, so BGE-M3 has the GPU to itself. Embeddings get cached to disk so you only do this once.

In [6]:
# print('Loading fine-tuned BGE-M3 (frozen base + LoRA adapter)...')
# ft_bi = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
# ft_bi.max_seq_length = 256
# inner = ft_bi[0].auto_model
# ft_bi[0].auto_model = PeftModel.from_pretrained(inner, BGE_M3_LORA_DIR, is_trainable=False)
# ft_bi[0].auto_model.eval()

# # Sanity check: confirm LoRA actually loaded
# frozen = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
# e_f = frozen.encode(['How do I prevent malaria?'], normalize_embeddings=True)
# e_t = ft_bi.encode(['How do I prevent malaria?'], normalize_embeddings=True)
# sim = float((e_f @ e_t.T)[0, 0])
# print(f'Cosine(frozen, fine-tuned): {sim:.4f}  (should be < 0.9999)')
# assert sim < 0.9999, 'LoRA did not load â€” embeddings identical to frozen base!'
# del frozen
# gc.collect(); torch.cuda.empty_cache()

In [7]:
# def encode_subset_indices(df, encoder, k):
#     out = {}
#     for g, grp in df.groupby(GCOL):
#         embs = encoder.encode(
#             grp[QCOL].tolist(),
#             normalize_embeddings=True, show_progress_bar=False,
#             batch_size=32, convert_to_numpy=True,
#         )
#         nn = NearestNeighbors(n_neighbors=min(k + 1, len(grp)), metric='cosine').fit(embs)
#         out[g] = {
#             'nn'       : nn,
#             'qs'       : np.array(grp[QCOL].astype(str).tolist(), dtype=object),
#             'ans'      : np.array(grp[ACOL].astype(str).tolist(), dtype=object),
#             'orig_idx' : np.array(grp.index.tolist()),
#         }
#     return out

# def retrieve_topk(df, indices, encoder, k, leave_self_out=False):
#     cands = [[] for _ in range(len(df))]
#     pos = {idx: i for i, idx in enumerate(df.index)}
#     for g, grp in df.groupby(GCOL):
#         m = indices.get(g) or next(iter(indices.values()))
#         qs = grp[QCOL].tolist()
#         embs = encoder.encode(qs, normalize_embeddings=True, show_progress_bar=False,
#                               batch_size=32, convert_to_numpy=True)
#         n_neighbors = min(k + 1, len(m['ans']))
#         _, idx_mat = m['nn'].kneighbors(embs, n_neighbors=n_neighbors)
#         for orig_idx, irow in zip(grp.index, idx_mat):
#             picked = []
#             for j in irow:
#                 if leave_self_out and m['orig_idx'][j] == orig_idx:
#                     continue
#                 picked.append({'q': str(m['qs'][j]), 'a': str(m['ans'][j])})
#                 if len(picked) >= k:
#                     break
#             cands[pos[orig_idx]] = picked
#     return cands

# K = CONFIG['top_k_context']

# # Train-only index for VAL (val should not see itself or other val items)
# print('Building train index...')
# t0 = time.time()
# train_idx = encode_subset_indices(train, ft_bi, K)
# print(f'  done in {time.time()-t0:.1f}s')

# # train+val index for TEST inference
# print('Building train+val index...')
# corpus = pd.concat([train, val], ignore_index=True).reset_index(drop=True)
# t0 = time.time()
# corpus_idx = encode_subset_indices(corpus, ft_bi, K)
# print(f'  done in {time.time()-t0:.1f}s')

In [8]:
# # Retrieve for train (leave-one-out), val, test
# print('Retrieving for train (leave-one-out)...')
# t0 = time.time()
# train_cands = retrieve_topk(train, train_idx, ft_bi, K, leave_self_out=True)
# print(f'  done in {time.time()-t0:.1f}s')

# print('Retrieving for val (from train, no self in train)...')
# t0 = time.time()
# val_cands = retrieve_topk(val, train_idx, ft_bi, K, leave_self_out=False)
# print(f'  done in {time.time()-t0:.1f}s')

# print('Retrieving for test (from train+val)...')
# t0 = time.time()
# test_cands = retrieve_topk(test, corpus_idx, ft_bi, K, leave_self_out=False)
# print(f'  done in {time.time()-t0:.1f}s')

# # Sanity peek
# print('\nSample train candidates:')
# i = 0
# print(f'  Q:    {train[QCOL].iloc[i][:120]}')
# print(f'  Gold: {train[ACOL].iloc[i][:120]}')
# for j, c in enumerate(train_cands[i]):
#     print(f'  Cand {j+1}: Q: {c["q"][:80]}')
#     print(f'           A: {c["a"][:80]}')

## 7 â€” Save Retrieval Results, Then Free BGE-M3

Saving lets you re-load these without re-running retrieval if training crashes.

In [9]:
# import pickle
# RETRIEVAL_CACHE = OUT_DIR / 'retrieval_cache.pkl'

# # with open(RETRIEVAL_CACHE, 'wb') as f:
# #     pickle.dump({
# #         'train_cands': train_cands,
# #         'val_cands'  : val_cands,
# #         'test_cands' : test_cands,
# #     }, f)
# # print(f'Saved retrieval cache to {RETRIEVAL_CACHE}')

# # # Free BGE-M3 from GPU
# # print(f'GPU before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
# # del ft_bi, train_idx, corpus_idx
# # gc.collect(); torch.cuda.empty_cache()
# # torch.cuda.synchronize()
# # print(f'GPU after cleanup:  {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

In [10]:
import pickle
RETRIEVAL_CACHE = OUT_DIR / 'retrieval_cache.pkl'

with open(RETRIEVAL_CACHE,'rb') as f:
    retrieved_dict = pickle.load(f)

train_cands,val_cands,test_cands = retrieved_dict['train_cands'],retrieved_dict['val_cands'],retrieved_dict['test_cands']

## 8 â€” Prompt Template

In [11]:
SUBSET_TO_LANGUAGE = {
    'Eng': 'English', 'Aka': 'Akan', 'Lug': 'Luganda',
    'Swa': 'Swahili', 'Amh': 'Amharic',
}
def language_of(subset):
    return SUBSET_TO_LANGUAGE.get(str(subset).split('_')[0], 'English')

def clip_text(text, max_chars):
    text = str(text).strip()
    if max_chars is None or len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(' ', 1)[0].strip()

def build_prompt(question: str, subset: str, candidates: list,
                 cand_max_chars: int = 320) -> str:
    """
    mT0-friendly prompt. Keep all 3 retrieved answer snippets, but compress retrieved
    questions so prompt budget goes to answer evidence instead of repeated question text.
    """
    lang = language_of(subset)
    parts = [f'Answer the following health question in {lang}.']
    parts.append(f'Question: {question}')
    if candidates:
        parts.append('')
        parts.append('Use these similar Q&A pairs as reference:')
        for i, c in enumerate(candidates, 1):
            q_trunc = clip_text(c['q'], CONFIG.get('cand_question_max_chars', 180))
            a_trunc = clip_text(c['a'], cand_max_chars)
            parts.append(f'{i}. Q: {q_trunc}')
            parts.append(f'   A: {a_trunc}')
    parts.append('')
    parts.append('Answer:')
    return '\n'.join(parts)

# Sanity peek
print(build_prompt(train[QCOL].iloc[0], train[GCOL].iloc[0],
                   train_cands[0], CONFIG['cand_answer_max_chars'])[:1500])
print('\n--- GOLD ---')
print(train[ACOL].iloc[0][:200])


Answer the following health question in Akan.
Question: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.

Use these similar Q&A pairs as reference:
1. Q: Ɔkwan bɛn so na mmabun betumi atew asiane a ɛwɔ hɔ sɛ wɔbɛtow ahyɛ wɔn so denam nsa anaa nnubɔne so, te sɛ sɛnea wɔbɛhwɛ sɛnea wɔnom nsa, wɔbɛn wɔn nnamfo a wogye wɔn di, na
   A: Wubetumi atew asiane a ɛwɔ nsa anaa nnubɔne mu basabasayɛ so denam nsa dodoɔ a wobɛnom a wobɛhwɛ na woakwati nsa pii a wobɛnom so, titiriw wɔ mmeae a wonnim hɔ yiye. Twa wo nnamfo a wogye wɔn di ho hyia na fa nhyehyɛeɛ a mobɛhwɛ mo ho mo ho so di dwuma. Gye nsa fi nkurɔfoɔ a wonnye wɔn nni hɔ da, na nom nsa a wohui sɛ
2. Q: Ɛdeɛn na mmabun bɛyɛ na wɔate aseɛ na wɔadi afoforo ahyeɛ so, a nea ɛka ho ne nsɛm a wɔka ne deɛ wɔnka, pene so nsrɛ, ne sɛ wo

## 9 â€” Load mT0-large in 4-bit + Attach LoRA

This is where the 6 GB savings come from. `BitsAndBytesConfig` with NF4 brings the 1.2B-param base model down to ~700 MB.

In [12]:
print(f'Loading tokenizer from {CONFIG["mt0_path"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['mt0_path'])

# 4-bit quantization config (QLoRA standard)
bnb_config = BitsAndBytesConfig(
    load_in_4bit                = True,
    bnb_4bit_quant_type         = 'nf4',          # NF4 is more accurate than fp4
    bnb_4bit_use_double_quant   = True,           # ~0.5 GB extra savings
    bnb_4bit_compute_dtype      = torch.float16,  
)

print(f'Loading mT0-large in 4-bit from {CONFIG["mt0_path"]} ...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    CONFIG['mt0_path'],
    quantization_config = bnb_config,
    device_map          = "auto",
    torch_dtype         = torch.float16,
)
print(f'Loaded.  GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading tokenizer from C:\Users\Papa Offei\Documents\lalang\mt0 ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=



Loading mT0-large in 4-bit from C:\Users\Papa Offei\Documents\lalang\mt0 ...


W0605 00:46:42.277000 26408 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\Papa Offei\AppData\Local\Programs\Python\Python313\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

c:\Users\Papa Offei\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded.  GPU allocated: 1.88 GB


In [13]:
from safetensors import safe_open

path = r"C:\Users\Papa Offei\Documents\lalang\mt0\model.safetensors"

with safe_open(path, framework="pt") as f:
    keys = list(f.keys())

print("Number of tensors:", len(keys))

for k in keys[:50]:
    print(k)

Number of tensors: 558
decoder.block.0.layer.0.SelfAttention.k.weight
decoder.block.0.layer.0.SelfAttention.o.weight
decoder.block.0.layer.0.SelfAttention.q.weight
decoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight
decoder.block.0.layer.0.SelfAttention.v.weight
decoder.block.0.layer.0.layer_norm.weight
decoder.block.0.layer.1.EncDecAttention.k.weight
decoder.block.0.layer.1.EncDecAttention.o.weight
decoder.block.0.layer.1.EncDecAttention.q.weight
decoder.block.0.layer.1.EncDecAttention.v.weight
decoder.block.0.layer.1.layer_norm.weight
decoder.block.0.layer.2.DenseReluDense.wi_0.weight
decoder.block.0.layer.2.DenseReluDense.wi_1.weight
decoder.block.0.layer.2.DenseReluDense.wo.weight
decoder.block.0.layer.2.layer_norm.weight
decoder.block.1.layer.0.SelfAttention.k.weight
decoder.block.1.layer.0.SelfAttention.o.weight
decoder.block.1.layer.0.SelfAttention.q.weight
decoder.block.1.layer.0.SelfAttention.v.weight
decoder.block.1.layer.0.layer_norm.weight
decoder.block.1.l

In [14]:

with safe_open(
    r"C:\Users\Papa Offei\Documents\lalang\mt0\model.safetensors",
    framework="pt"
) as f:
    print("shared.weight" in f.keys())

True


In [15]:
# Prep for QLoRA training (casts norm layers to fp32 for stability)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=CONFIG['gradient_checkpointing']
)

lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    inference_mode = False,
    r              = CONFIG['lora_r'],
    lora_alpha     = CONFIG['lora_alpha'],
    lora_dropout   = CONFIG['lora_dropout'],
    target_modules = CONFIG['lora_target_modules'],
    bias           = 'none',
)
model = get_peft_model(model, lora_config)

# This is the crucial line â€” required when LoRA + gradient checkpointing combine
model.enable_input_require_grads()

model.print_trainable_parameters()
print(f'GPU allocated after LoRA: {torch.cuda.memory_allocated()/1e9:.2f} GB')

trainable params: 4,718,592 || all params: 1,234,299,904 || trainable%: 0.3823
GPU allocated after LoRA: 2.92 GB


## 10 â€” Tokenize Training Data

In [16]:
def tokenize_example(question, subset, candidates, gold_answer=None):
    prompt = build_prompt(question, subset, candidates, CONFIG['cand_answer_max_chars'])
    enc = tokenizer(
        prompt,
        max_length=CONFIG['max_input_length'],
        truncation=True, padding=False,
    )
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    if gold_answer is not None:
        labels = tokenizer(
            text_target = gold_answer,
            max_length  = CONFIG['max_target_length'],
            truncation  = True, padding = False,
        )
        out['labels'] = [
            tok if tok != tokenizer.pad_token_id else -100
            for tok in labels['input_ids']
        ]
    return out

def stratified_indices(df, n, group_col=GCOL, seed=SEED):
    """Sample rows across all subsets while keeping the subset mix close to the original."""
    if n is None or n >= len(df):
        return list(range(len(df)))
    rng = np.random.default_rng(seed)
    counts = df[group_col].value_counts().sort_index()
    raw = counts / counts.sum() * n
    take = np.floor(raw).astype(int).clip(lower=1)
    remainder = int(n - take.sum())
    if remainder > 0:
        for subset in (raw - take).sort_values(ascending=False).index[:remainder]:
            take.loc[subset] += 1
    elif remainder < 0:
        for subset in take.sort_values(ascending=False).index[:abs(remainder)]:
            if take.loc[subset] > 1:
                take.loc[subset] -= 1
    chosen = []
    for subset, k in take.items():
        idx = df.index[df[group_col] == subset].to_numpy()
        chosen.extend(rng.choice(idx, size=min(int(k), len(idx)), replace=False).tolist())
    rng.shuffle(chosen)
    return chosen

def subset_rows(df, cands, n, seed=SEED):
    idx = stratified_indices(df, n, seed=seed)
    return df.iloc[idx].reset_index(drop=True), [cands[i] for i in idx], idx

def df_to_hf_dataset(df, cands, with_labels=True):
    records = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc='Tokenizing'):
        rec = tokenize_example(
            question    = row[QCOL],
            subset      = row[GCOL],
            candidates  = cands[i],
            gold_answer = row[ACOL] if with_labels and ACOL in df.columns else None,
        )
        records.append(rec)
    return Dataset.from_list(records)

if CONFIG['quick_experiment']:
    train_model_df, train_model_cands, train_model_idx = subset_rows(
        train, train_cands, CONFIG['quick_train_rows'], seed=SEED
    )
    print(f'Quick experiment: training on {len(train_model_df):,}/{len(train):,} stratified rows')
    print(train_model_df[GCOL].value_counts().sort_index().to_string())
else:
    train_model_df, train_model_cands = train, train_cands

print('Tokenizing TRAIN ...')
train_ds = df_to_hf_dataset(train_model_df, train_model_cands, with_labels=True)

# Tiny loss-only eval set if eval_during_training is enabled.
SMALL_EVAL_N = min(200, len(val))
small_val_df, small_val_cands, small_val_idx = subset_rows(val, val_cands, SMALL_EVAL_N, seed=SEED + 1)
small_val_ds = df_to_hf_dataset(small_val_df, small_val_cands, with_labels=True)

print(f'\nTrain dataset: {train_ds}')
print(f'Small eval:    {small_val_ds}')

lens = [len(x) for x in train_ds['input_ids']]
label_lens = [sum(1 for tok in x if tok != -100) for x in train_ds['labels']]
print(f'\nInput token lengths - mean: {np.mean(lens):.0f}, p50: {np.percentile(lens, 50):.0f}, '
      f'p95: {np.percentile(lens, 95):.0f}, max: {max(lens)}')
print(f'Inputs hitting the max ({CONFIG["max_input_length"]}): '
      f'{sum(1 for l in lens if l >= CONFIG["max_input_length"])} '
      f'({100 * sum(1 for l in lens if l >= CONFIG["max_input_length"]) / len(lens):.1f}%)')
print(f'Target token lengths - mean: {np.mean(label_lens):.0f}, p50: {np.percentile(label_lens, 50):.0f}, '
      f'p95: {np.percentile(label_lens, 95):.0f}, max: {max(label_lens)}')
print(f'Targets hitting the max ({CONFIG["max_target_length"]}): '
      f'{sum(1 for l in label_lens if l >= CONFIG["max_target_length"])} '
      f'({100 * sum(1 for l in label_lens if l >= CONFIG["max_target_length"]) / len(label_lens):.1f}%)')


Quick experiment: training on 8,000/29,814 stratified rows
subset
Aka_Gha    1195
Amh_Eth     495
Eng_Eth    1051
Eng_Gha    1192
Eng_Ken     558
Eng_Uga    2046
Lug_Uga     908
Swa_Ken     555
Tokenizing TRAIN ...


Tokenizing:   0%|          | 0/8000 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/200 [00:00<?, ?it/s]


Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})
Small eval:    Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})

Input token lengths - mean: 334, p50: 358, p95: 384, max: 384
Inputs hitting the max (384): 3060 (38.2%)
Target token lengths - mean: 131, p50: 117, p95: 240, max: 240
Targets hitting the max (240): 1625 (20.3%)


## 11 â€” Train

Settings tuned for ~10-14 hours on RTX 4050 6 GB.

**Critical for stability:**
- fp16 compute (T4-style hardware quirk doesn't apply on RTX 4050 â€” bf16 works too but fp16 is faster)
- `prepare_model_for_kbit_training` casts layer norms to fp32 (already done)
- `enable_input_require_grads` so gradient checkpointing + LoRA cooperate
- 4-bit base + fp16 LoRA + fp32 layer norms = the QLoRA recipe

In [17]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

# RTX 4050 (Ada) supports bf16; prefer it for stability with QLoRA
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f'Mixed precision: bf16={use_bf16}, fp16={use_fp16}')

EVAL_DURING_TRAINING = bool(CONFIG['eval_during_training'])

training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(OUT_DIR),
    num_train_epochs            = CONFIG['num_epochs'],
    per_device_train_batch_size = CONFIG['batch_size'],
    per_device_eval_batch_size  = CONFIG['batch_size'],
    gradient_accumulation_steps = CONFIG['gradient_accumulation'],
    learning_rate               = CONFIG['learning_rate'],
    warmup_ratio                = CONFIG['warmup_ratio'],
    weight_decay                = CONFIG['weight_decay'],
    fp16                        = use_fp16,
    bf16                        = use_bf16,
    gradient_checkpointing      = CONFIG['gradient_checkpointing'],
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    logging_steps               = 25,
    eval_strategy               = 'epoch' if EVAL_DURING_TRAINING else 'no',
    save_strategy               = 'epoch' if EVAL_DURING_TRAINING else 'no',
    save_total_limit            = 1,
    load_best_model_at_end      = EVAL_DURING_TRAINING,
    metric_for_best_model       = 'eval_loss' if EVAL_DURING_TRAINING else None,
    greater_is_better           = False,
    report_to                   = 'none',
    predict_with_generate       = False,
    dataloader_num_workers      = 0,    # Windows - 0 is safer than >0
    dataloader_pin_memory       = True,
    remove_unused_columns       = False,
    optim                       = 'adamw_torch',       # LoRA params are small; avoids bnb 8-bit CUDA 13.2 issue
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = small_val_ds if EVAL_DURING_TRAINING else None,
    processing_class = tokenizer,
    data_collator    = data_collator,
)

print(f'Training on {len(train_ds):,} examples for {CONFIG["num_epochs"]} epoch(s)')
print(f'Effective batch size = {CONFIG["batch_size"] * CONFIG["gradient_accumulation"]}')
print(f'Optimizer steps per epoch ~ {len(train_ds) // (CONFIG["batch_size"] * CONFIG["gradient_accumulation"])}')
print(f'Full-run reference would be ~{len(train) // (CONFIG["batch_size"] * CONFIG["gradient_accumulation"])} optimizer steps per epoch')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Mixed precision: bf16=True, fp16=False
Training on 8,000 examples for 1 epoch(s)
Effective batch size = 16
Optimizer steps per epoch ~ 500
Full-run reference would be ~1863 optimizer steps per epoch


In [18]:
# Start training. This is the ~10-14 hour cell.
t0 = time.time()
trainer.train()
print(f'\nTraining complete in {(time.time()-t0)/3600:.1f} hours')

model.save_pretrained(str(OUT_DIR / 'final'))
tokenizer.save_pretrained(str(OUT_DIR / 'final'))
print(f'Saved to {OUT_DIR / "final"}')

Step,Training Loss
25,36.287922
50,31.483118
75,31.535569
100,29.754062
125,30.515347
150,30.595435
175,28.621409
200,28.029976
225,29.056367
250,27.343086



Training complete in 3.4 hours
Saved to mt0-rag-finetuned\final


In [19]:
# Check training loss curve â€” must go DOWN and not be NaN
print('Training log (last 20 entries):')
for entry in trainer.state.log_history[-20:]:
    if 'loss' in entry or 'eval_loss' in entry:
        print(entry)

Training log (last 20 entries):
{'loss': 31.48311767578125, 'grad_norm': 3.9914190769195557, 'learning_rate': 0.0002848421052631579, 'epoch': 0.1, 'step': 50}
{'loss': 31.53556884765625, 'grad_norm': 4.880775451660156, 'learning_rate': 0.00026905263157894733, 'epoch': 0.15, 'step': 75}
{'loss': 29.7540625, 'grad_norm': 9.59973430633545, 'learning_rate': 0.0002532631578947368, 'epoch': 0.2, 'step': 100}
{'loss': 30.5153466796875, 'grad_norm': 5.34889030456543, 'learning_rate': 0.0002374736842105263, 'epoch': 0.25, 'step': 125}
{'loss': 30.5954345703125, 'grad_norm': 5.2865118980407715, 'learning_rate': 0.0002216842105263158, 'epoch': 0.3, 'step': 150}
{'loss': 28.62140869140625, 'grad_norm': 6.07497501373291, 'learning_rate': 0.00020589473684210524, 'epoch': 0.35, 'step': 175}
{'loss': 28.0299755859375, 'grad_norm': 4.880159378051758, 'learning_rate': 0.0001901052631578947, 'epoch': 0.4, 'step': 200}
{'loss': 29.0563671875, 'grad_norm': 5.010234832763672, 'learning_rate': 0.000174315789

## 12 â€” Quick Sanity Check on 5 Val Examples Before Full Eval

If outputs look like garbage here, **stop and inspect** before spending an hour on full val generation.

In [20]:
model.eval()

@torch.no_grad()
def generate_for(df, cands, indices, num_beams=4, max_length=None):
    max_length = max_length or CONFIG['max_target_length']
    results = []
    for i in indices:
        prompt = build_prompt(df[QCOL].iloc[i], df[GCOL].iloc[i], cands[i],
                              CONFIG['cand_answer_max_chars'])
        enc = tokenizer(
            prompt, max_length=CONFIG['max_input_length'],
            truncation=True, return_tensors='pt'
        ).to(DEVICE)
        out = model.generate(
            **enc,
            max_length          = max_length,
            num_beams           = num_beams,
            no_repeat_ngram_size= CONFIG['no_repeat_ngram_size'],
            length_penalty      = CONFIG['length_penalty'],
            early_stopping      = True,
        )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=False)[0]
        results.append(decoded)
    return results

sample_idx = list(range(5))
raw_outputs = generate_for(val, val_cands, sample_idx, num_beams=4)

for k, i in enumerate(sample_idx):
    print(f'=== Row {i} ({val[GCOL].iloc[i]}) ===')
    print(f'  Q:    {val[QCOL].iloc[i][:120]}')
    print(f'  Gold: {val[ACOL].iloc[i][:200]}')
    print(f'  Pred: {raw_outputs[k][:200]}')
    print()

=== Row 0 (Aka_Gha) ===
  Q:    Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn a w
  Gold: Nhyehyɛeɛ aa ama ne mu so te sɛ senea aborɔfo ka no 'STEM' ne 'vocational training' se ɛbɛ adrɛse mmabun kuokuo ahohia soronko ne ɔhaw. Nea wokaho yɛ: atufrafo a bronni bɛkasɛ 'refugee anaa immigrant'
  Pred: <pad> Aban adwumakuw, ahyehyɛde ahorow a wɔnyɛ adwuma a ɛde hwehwɛ mfaso, ne ankorankoro nnwumayɛbea ahorow betumi di dwuma titiriw wɔ sika a de ma na wɔboa mmabun a wohia neaɛma sokoronko ne ohaw aho

=== Row 1 (Aka_Gha) ===
  Q:    Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?
  Gold: Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abusuabɔ ho. Kamfo wɔn hokwan ahorow ne nna ne awo akwahosan ho nhyehyɛe ne nsɛm a ɛho hia a wobenya. B
  Pred: <pad> Nneɛma pii a ɛbɛboa mmabun ma wɔate wɔn nna ne awo hokwan ahorow ase na wɔakamfo wɔn, a na ɛka ho ne: Intan

## 13 â€” Full Val Generation & Evaluation

Estimated time on RTX 4050: ~1.5-2 hours for ~6700 val rows at beam=4. If you need it faster, drop `num_beams` to 2.

In [21]:
@torch.no_grad()
def generate_answers(df, cands, batch_size=4, num_beams=4):
    answers = []
    for start in tqdm(range(0, len(df), batch_size), desc='Generating'):
        end = min(start + batch_size, len(df))
        prompts = [
            build_prompt(df[QCOL].iloc[i], df[GCOL].iloc[i], cands[i],
                         CONFIG['cand_answer_max_chars'])
            for i in range(start, end)
        ]
        enc = tokenizer(
            prompts,
            max_length=CONFIG['max_input_length'],
            truncation=True, padding=True, return_tensors='pt',
        ).to(DEVICE)
        out = model.generate(
            **enc,
            max_length          = CONFIG['max_target_length'],
            num_beams           = num_beams,
            no_repeat_ngram_size= CONFIG['no_repeat_ngram_size'],
            length_penalty      = CONFIG['length_penalty'],
            early_stopping      = True,
        )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        # Belt-and-suspenders: strip any sentinel residue (shouldn't appear for mT0 but cheap)
        decoded = [re.sub(r'<extra_id_\d+>', '', d).strip() for d in decoded]
        answers.extend(decoded)
    return answers

if CONFIG['quick_experiment']:
    val_eval_df, val_eval_cands, val_eval_idx = subset_rows(
        val, val_cands, CONFIG['quick_eval_rows'], seed=SEED + 2
    )
    print(f'Quick experiment: generating val sample ({len(val_eval_df)}/{len(val)} rows)')
else:
    val_eval_df, val_eval_cands = val, val_cands

val_eval_top1 = [c[0]['a'] if c else '' for c in val_eval_cands]
print(f'Generating validation answers ({len(val_eval_df)} rows, beams={CONFIG["num_beams"]})...')
t0 = time.time()
val_preds = generate_answers(
    val_eval_df, val_eval_cands,
    batch_size=CONFIG['gen_batch_size'],
    num_beams=CONFIG['num_beams'],
)
print(f'  done in {(time.time()-t0)/60:.1f} min')

# Save val predictions as a checkpoint in case anything happens later
val_pred_payload = {
    'quick_experiment': CONFIG['quick_experiment'],
    'num_rows': len(val_eval_df),
    'num_beams': CONFIG['num_beams'],
    'ids': val_eval_df[IDCOL].tolist(),
    'predictions': val_preds,
}
with open(OUT_DIR / 'val_preds.json', 'w', encoding='utf-8') as f:
    json.dump(val_pred_payload, f, ensure_ascii=False)
print(f'  saved to {OUT_DIR / "val_preds.json"}')


Quick experiment: generating val sample (800/6686 rows)
Generating validation answers (800 rows, beams=1)...


Generating:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  done in 73.2 min
  saved to mt0-rag-finetuned\val_preds.json


In [22]:
baseline_overall = rouge_metrics(val_eval_top1, val_eval_df[ACOL].tolist())
overall = rouge_metrics(val_preds, val_eval_df[ACOL].tolist())
print(f'\n=== mT0-large + RAG - Val {"sample" if CONFIG["quick_experiment"] else "full"} ===')
print(f'   rows              : {len(val_eval_df)}')
print(f'   top1 scaffold R1  : {baseline_overall["rouge1_f1"]:.4f}')
print(f'   mT0 ROUGE-1 F1    : {overall["rouge1_f1"]:.4f}')
print(f'   mT0 ROUGE-L F1    : {overall["rougeL_f1"]:.4f}')
print(f'   delta vs scaffold : {overall["rouge1_f1"] - baseline_overall["rouge1_f1"]:+.4f}')

print('\nPer subset - retrieved top1 scaffold:')
baseline_report = per_subset_report(val_eval_top1, val_eval_df[ACOL].tolist(), val_eval_df[GCOL].tolist(), 'top1')
print(baseline_report.to_string(index=False))

print('\nPer subset - mT0 generated:')
report = per_subset_report(val_preds, val_eval_df[ACOL].tolist(), val_eval_df[GCOL].tolist(), 'mt0_rag')
print(report.to_string(index=False))

subset_cmp = baseline_report[baseline_report['subset'] != 'OVERALL'][['subset', 'top1_r1']].merge(
    report[report['subset'] != 'OVERALL'][['subset', 'mt0_rag_r1']], on='subset', how='outer'
)
subset_cmp['delta_vs_top1'] = (subset_cmp['mt0_rag_r1'] - subset_cmp['top1_r1']).round(4)
print('\nSubset deltas vs same-sample top1 scaffold:')
print(subset_cmp.sort_values('delta_vs_top1', ascending=False).to_string(index=False))



=== mT0-large + RAG - Val sample ===
   rows              : 800
   top1 scaffold R1  : 0.5435
   mT0 ROUGE-1 F1    : 0.4136
   mT0 ROUGE-L F1    : 0.3772
   delta vs scaffold : -0.1299

Per subset - retrieved top1 scaffold:
 subset   n  top1_r1  top1_rL
Aka_Gha 133   0.2816   0.1693
Amh_Eth  55   0.1846   0.1726
Eng_Eth  68   0.5789   0.5552
Eng_Gha 132   0.2786   0.1798
Eng_Ken  47   0.7702   0.7500
Eng_Uga 202   0.8391   0.8251
Lug_Uga 101   0.5177   0.4928
Swa_Ken  62   0.8556   0.8402
OVERALL 800   0.5435   0.4966

Per subset - mT0 generated:
 subset   n  mt0_rag_r1  mt0_rag_rL
Aka_Gha 133      0.2579      0.1801
Amh_Eth  55      0.1849      0.1720
Eng_Eth  68      0.5928      0.5759
Eng_Gha 132      0.2737      0.2038
Eng_Ken  47      0.5823      0.5687
Eng_Uga 202      0.5587      0.5429
Lug_Uga 101      0.3166      0.2939
Swa_Ken  62      0.6096      0.5835
OVERALL 800      0.4136      0.3772

Subset deltas vs same-sample top1 scaffold:
 subset  top1_r1  mt0_rag_r1  delta_vs_to

## 14 â€” Compare Against G (your previous best: 0.5516)

In [23]:
prev_G_overall = 0.5516
prev_G_per_subset = {
    'Aka_Gha': 0.3127, 'Amh_Eth': 0.1740, 'Eng_Eth': 0.6384,
    'Eng_Gha': 0.2996, 'Eng_Ken': 0.7992, 'Eng_Uga': 0.8231,
    'Lug_Uga': 0.5587, 'Swa_Ken': 0.7941,
}

mt0_per_subset = (report[report['subset'] != 'OVERALL']
                  .set_index('subset')['mt0_rag_r1'])

cmp = pd.DataFrame({
    'G prev full-val best': prev_G_per_subset,
    'mT0 + RAG'          : mt0_per_subset,
})
cmp['delta_vs_G_reference'] = (cmp['mT0 + RAG'] - cmp['G prev full-val best']).round(4)
print(cmp.sort_values('delta_vs_G_reference', ascending=False))

if CONFIG['quick_experiment']:
    print('\nDecision rule for this quick run:')
    print('1. First check delta_vs_top1 above; if mT0 cannot beat the retrieved answer scaffold, stop/tune prompts.')
    print('2. If only a few subsets improve, use mT0 selectively on those subsets instead of replacing G globally.')
    print('3. G reference numbers above are full-val, so use them only as a rough target until quick_experiment=False.')
else:
    print(f'\nOverall: G = {prev_G_overall:.4f}   mT0+RAG = {overall["rouge1_f1"]:.4f}   '
          f'delta = {overall["rouge1_f1"] - prev_G_overall:+.4f}')


         G prev full-val best  mT0 + RAG  delta_vs_G_reference
Amh_Eth                0.1740     0.1849                0.0109
Eng_Gha                0.2996     0.2737               -0.0259
Eng_Eth                0.6384     0.5928               -0.0456
Aka_Gha                0.3127     0.2579               -0.0548
Swa_Ken                0.7941     0.6096               -0.1845
Eng_Ken                0.7992     0.5823               -0.2169
Lug_Uga                0.5587     0.3166               -0.2421
Eng_Uga                0.8231     0.5587               -0.2644

Decision rule for this quick run:
1. First check delta_vs_top1 above; if mT0 cannot beat the retrieved answer scaffold, stop/tune prompts.
2. If only a few subsets improve, use mT0 selectively on those subsets instead of replacing G globally.
3. G reference numbers above are full-val, so use them only as a rough target until quick_experiment=False.


## 15 â€” Generate Test Predictions & Submit

In [24]:
print(f'Generating test ({len(test)} rows)...')
t0 = time.time()
test_preds = generate_answers(
    test, test_cands,
    batch_size=CONFIG['gen_batch_size'],
    num_beams=CONFIG['num_beams'],
)
print(f'  done in {(time.time()-t0)/60:.1f} min')

with open(OUT_DIR / 'test_preds.json', 'w', encoding='utf-8') as f:
    json.dump(test_preds, f, ensure_ascii=False)

Generating test (2618 rows)...


Generating:   0%|          | 0/655 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
clean = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in test_preds]
sub = pd.DataFrame({
    'ID'        : test[IDCOL],
    'TargetRLF1': clean,
    'TargetR1F1': clean,
    'TargetLLM' : clean,
})[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
assert len(sub) == len(test)
sub.to_csv(SUBMISSION, index=False, encoding='utf-8')
print(f'Saved {SUBMISSION} ({len(sub)} rows)')
display(sub.head(3))

In [ ]:
# Optional: hybrid with G â€” use mT0 only on subsets where it beat G on val
# Edit G_SUB_PATH and uncomment.
# G_SUB_PATH = './submission_best_per_subset.csv'
# 
# g_sub = pd.read_csv(G_SUB_PATH)
# g_pred = dict(zip(g_sub['ID'], g_sub['TargetR1F1']))
# 
# USE_MT0 = {s for s in cmp.index if cmp.loc[s, 'delta'] > 0}
# print(f'Using mT0 on: {sorted(USE_MT0)}')
# 
# hybrid_preds = [
#     clean[i] if test[GCOL].iloc[i] in USE_MT0 else g_pred[test[IDCOL].iloc[i]]
#     for i in range(len(test))
# ]
# sub_h = pd.DataFrame({
#     'ID': test[IDCOL],
#     'TargetRLF1': hybrid_preds,
#     'TargetR1F1': hybrid_preds,
#     'TargetLLM' : hybrid_preds,
# })
# sub_h.to_csv('./submission_mt0_hybrid.csv', index=False, encoding='utf-8')
# print('Saved hybrid submission.')

## 16 â€” Troubleshooting

### If training crashes with OOM on the first step
- `max_input_length=320`, `max_target_length=200`
- `cand_answer_max_chars=250`
- `lora_target_modules=['q', 'v']` (already set)
- Reduce `top_k_context` from 3 to 2

### If loss is NaN
- Switch `bf16=True, fp16=False` explicitly in training_args (RTX 4050 supports bf16 natively)
- Lower learning rate to `2e-4`
- Make sure `prepare_model_for_kbit_training` was called (cell 9)

### If sanity outputs in section 12 look like garbage
- Check that `MT0_PATH` actually points to **mT0** not mT5 â€” mT5 would emit `<extra_id_0>` everywhere
- Check `tokenizer.decode(train_ds[0]['input_ids'])` shows your prompt template correctly
- Inspect `trainer.state.log_history` â€” loss should go from ~5 down to ~1.5-2.5

### If full val generation is too slow (>3h)
- `num_beams=2` (about 2Ã— speedup, small ROUGE cost)
- `gen_batch_size=8` if VRAM allows
- Generate val on a 1000-row sample first to confirm gains before committing to full

### If the model doesn't beat G (0.5516)
- Check that retrieval candidates are actually in the right language (cell 6 sanity)
- Try `length_penalty=1.5` to encourage longer outputs (ROUGE-1 favors length up to a point)
- Try `num_beams=6` and `no_repeat_ngram_size=4`

### Resume from a checkpoint
- `trainer.train(resume_from_checkpoint=True)` picks up from the last saved checkpoint in `OUT_DIR`

### Re-use the retrieval cache (if you crash and restart)
```python
import pickle
with open(OUT_DIR / 'retrieval_cache.pkl', 'rb') as f:
    cache = pickle.load(f)
train_cands = cache['train_cands']
val_cands   = cache['val_cands']
test_cands  = cache['test_cands']
```